# Detector de quedas: treino da LSTM (WEDA-FALL + Apple Watch)

Este notebook treina a rede neural do projeto e exporta o modelo que a API usa.

**O ciclo completo:**

1. Montar o dataset com o pré-processamento corrigido (o **mesmo** código que a API usa: `app/queda/sinais.py`)
2. Treinar um **modelo de referência** simples, que serve de régua
3. Treinar a **LSTM**
4. Escolher o **limiar** na validação e medir no **teste** (pessoas que a rede nunca viu)
5. **Quality gate**: só exporta se a LSTM passar nos critérios mínimos
6. Registrar o experimento no **MLflow** e **exportar** a pasta do modelo

> Ambiente: Colab com ou sem GPU. Sem GPU o treino leva alguns minutos.

## 1. Baixar o projeto

O notebook importa os módulos de `app/queda/`, então precisa do repositório. Troque `BRANCH` se o código estiver em outra branch.

In [ ]:
BRANCH = "feat/detector-queda-apple-watch"
REPO = "https://github.com/Grupo-Up/second-challenge.git"

![ -d second-challenge ] || git clone --depth 1 -b {BRANCH} {REPO}
%cd second-challenge
!pip install -q mlflow

## 2. (Opcional) Gravações do Apple Watch

Se o grupo já gravou com o **Sensor Logger**, coloque as gravações no Google Drive com **uma pasta por gravação**, no padrão `CLASSE_USUARIO_REPETICAO`, cada uma com o `WristMotion.csv` exportado dentro:

```
MyDrive/gravacoes_apple_watch/
    F01_U40_R01/WristMotion.csv     <- queda (F01..F08), adulto U40
    D01_U40_R01/WristMotion.csv     <- atividade (D01..D11)
    D03_I01_R01/WristMotion.csv     <- usuário começando com I = idoso
```

Use os mesmos códigos de atividade e queda do WEDA (lista na célula seguinte). **Quedas: só adultos jovens, sobre colchão.** Idosos gravam apenas atividades do dia a dia, com consentimento.

Sem gravações, deixe `PASTA_APPLE_WATCH = None`: o treino usa só o WEDA.

In [ ]:
PASTA_APPLE_WATCH = None  # ex.: "/content/drive/MyDrive/gravacoes_apple_watch"

# Usuarios do Apple Watch que vao SO para o teste (nunca para o treino).
# E so com eles que se sabe se a rede funciona no relogio de voces.
TESTE_APPLE_WATCH = []    # ex.: ["apple_watch:U41", "apple_watch:I01"]

if PASTA_APPLE_WATCH:
    from google.colab import drive
    drive.mount("/content/drive")

CLASSES = {
    "F01": "Queda para a frente andando (escorregão)", "F02": "Queda lateral andando (escorregão)",
    "F03": "Queda para trás andando (escorregão)", "F04": "Queda para a frente andando (tropeço)",
    "F05": "Queda para trás ao tentar sentar", "F06": "Queda para a frente sentado (desmaio)",
    "F07": "Queda para trás sentado (desmaio)", "F08": "Queda lateral sentado (desmaio)",
    "D01": "Andar", "D02": "Correr", "D03": "Subir e descer escada",
    "D04": "Sentar, esperar e levantar", "D05": "Tentar levantar e cair de volta na cadeira",
    "D06": "Agachar, amarrar o sapato e levantar", "D07": "Tropeçar andando (sem cair)",
    "D08": "Pular de leve (pegar algo no alto)", "D09": "Bater a mão na mesa",
    "D10": "Bater palmas", "D11": "Abrir e fechar porta",
}

## 3. Montar o dataset

O que mudou em relação ao `extracao.py` antigo (detalhes em `app/queda/dados.py`):

| Problema antigo | Correção |
|---|---|
| `id_ = "U01_R01"` se repetia nas 19 pastas: 969 gravações viravam 56 sequências misturadas | identificador = classe + usuário + repetição |
| junção pelo instante exato + `dropna` jogava fora ~98,6% das linhas | reamostragem numa grade comum de 50 Hz |
| quedas a ~40 Hz e atividades a ~50 Hz: a rede podia aprender a frequência | tudo reamostrado para 50 Hz |
| scaler ajustado antes de separar treino e teste | média e desvio só do treino |
| divisão aleatória de janelas | divisão **por pessoa** |

Cada janela tem 2,5 s (125 amostras) e 4 sinais que não dependem do relógio nem do pulso: \(|a|\), \(|\omega|\), \(a_v\) (aceleração na direção da gravidade) e \(a_h\) (no plano horizontal).

In [ ]:
import numpy as np, pandas as pd
from app.queda import dados, sinais, avaliacao, referencia, exportar

gravacoes = dados.listar_weda("app/rede-neural/dataset")
if PASTA_APPLE_WATCH:
    gravacoes += dados.listar_apple_watch(PASTA_APPLE_WATCH)

X, y, info = dados.montar(gravacoes)
info["parte"] = dados.separar(
    info,
    teste=dados.TESTE_WEDA + TESTE_APPLE_WATCH,
    validacao=dados.VALIDACAO_WEDA,
)
info["y"] = y

print(f"{len(gravacoes)} gravações -> {len(X)} janelas de forma {X.shape[1:]}")
display(info.groupby(["parte", "origem", "idoso"])["y"].agg(janelas="size", quedas="sum"))

In [ ]:
import matplotlib.pyplot as plt

def mostrar(classe, usuario="U01", repeticao="R01"):
    g = next(g for g in gravacoes if g.classe == classe and g.usuario == usuario and g.repeticao == repeticao)
    s = dados.ler(g)
    t = np.arange(len(s)) / sinais.FREQ_HZ
    fig, eixos = plt.subplots(1, 2, figsize=(12, 2.8), sharex=True)
    eixos[0].plot(t, s[:, 0]); eixos[0].set_title(f"{classe} - {CLASSES[classe]}: |a| (m/s²)")
    eixos[1].plot(t, s[:, 1], color="C1"); eixos[1].set_title("|ω| (rad/s)")
    for e in eixos: e.set_xlabel("tempo (s)")
    plt.tight_layout(); plt.show()

mostrar("F01")   # queda: pico forte de |a| no impacto
mostrar("D01")   # andar: oscilação regular em torno de 9,8
mostrar("D09")   # bater na mesa: também tem pico. É a atividade que mais confunde

## 4. Modelo de referência (a régua)

Estatísticas simples de cada janela + árvores de decisão. **Não é a rede neural do projeto**: serve para responder "a LSTM vale a pena?". Se a LSTM não ganhar dele, a complexidade não se paga.

O limiar é escolhido **na validação** (o maior que ainda detecta ≥ 95% das quedas) e só depois aplicado ao teste.

In [ ]:
treino = (info["parte"] == "treino").to_numpy()
validacao = (info["parte"] == "validacao").to_numpy()
teste = (info["parte"] == "teste").to_numpy()
RECALL_MINIMO = 0.95

ref = referencia.ModeloReferencia(referencia.treinar(X[treino], y[treino]))
limiar_ref = avaliacao.escolher_limiar(info[validacao], y[validacao], ref.prever(X[validacao]), RECALL_MINIMO)
metricas_ref = avaliacao.resumo(info[teste], y[teste], ref.prever(X[teste]), limiar_ref)
pd.Series(metricas_ref).round(3)

## 5. LSTM

Arquitetura parecida com a que o grupo já tinha, com duas mudanças: **uma saída** (queda ou não queda, sigmoide) em vez de 19 classes, porque o alerta só precisa dessa resposta; e **peso maior para a classe queda**, porque ela é rara (~7% das janelas).

Padronização: \( x' = \frac{x - \mu_{\text{treino}}}{\sigma_{\text{treino}}} \), com \(\mu\) e \(\sigma\) calculados **só no treino** e salvos junto com o modelo.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)

media = X[treino].reshape(-1, X.shape[2]).mean(axis=0)
desvio = X[treino].reshape(-1, X.shape[2]).std(axis=0)
Xp = sinais.padronizar(X, media, desvio)

n_pos, n_neg = int(y[treino].sum()), int((y[treino] == 0).sum())
pesos = {0: 1.0, 1: n_neg / n_pos}

HIPER = dict(unidades_lstm=64, dropout=0.3, densa=32, taxa_aprendizado=1e-3, lote=64, epocas_max=40)

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(sinais.AMOSTRAS_JANELA, len(sinais.CANAIS))),
    tf.keras.layers.LSTM(HIPER["unidades_lstm"]),
    tf.keras.layers.Dropout(HIPER["dropout"]),
    tf.keras.layers.Dense(HIPER["densa"], activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(HIPER["taxa_aprendizado"]),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.Precision(name="precisao"),
             tf.keras.metrics.AUC(name="auc")],
)
historico = modelo.fit(
    Xp[treino], y[treino],
    validation_data=(Xp[validacao], y[validacao]),
    epochs=HIPER["epocas_max"], batch_size=HIPER["lote"], class_weight=pesos,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6,
                                                restore_best_weights=True)],
    verbose=2,
)

In [ ]:
pd.DataFrame(historico.history)[["loss", "val_loss", "val_recall", "val_precisao"]].plot(figsize=(8, 3), grid=True);

## 6. Limiar na validação, resultado no teste

As métricas que importam para a família (ver `app/queda/avaliacao.py`):

- **recall de quedas**: de cada 100 quedas, quantas geraram alerta;
- **alarmes falsos por hora** de atividade, e só nos **idosos**.

> Atenção ao ler "alarmes falsos por hora": as atividades do WEDA foram escolhidas para confundir (tropeçar, pular, bater na mesa). Uma hora de WEDA é muito mais difícil que uma hora de vida real, e a amostra é pequena (cerca de 0,4 h no teste).

In [ ]:
prever_lstm = lambda x: modelo.predict(sinais.padronizar(x, media, desvio), verbose=0).reshape(-1)

limiar = avaliacao.escolher_limiar(info[validacao], y[validacao], prever_lstm(X[validacao]), RECALL_MINIMO)
p_teste = prever_lstm(X[teste])
metricas = avaliacao.resumo(info[teste], y[teste], p_teste, limiar)

comparacao = pd.DataFrame({"referência": metricas_ref, "LSTM": metricas}).round(3)
display(comparacao)

# Onde a LSTM erra: alarmes falsos por atividade
erros = info[teste].assign(alerta=p_teste >= limiar)
display(erros[erros["y"] == 0].groupby("classe")["alerta"].mean().rename(index=CLASSES).round(3)
        .to_frame("fração de janelas com alarme"))

# Se houver gravacoes do Apple Watch no teste, o numero que mais importa
aw = (info["origem"] == "apple_watch").to_numpy() & teste
if aw.any():
    print("Só Apple Watch:", avaliacao.resumo(info[aw], y[aw], prever_lstm(X[aw]), limiar))

## 7. Quality gate

Critérios mínimos para o modelo ir para a API (ajustem com o grupo e registrem a decisão):

1. detectar pelo menos 90% das quedas do teste;
2. não ser pior que o modelo de referência em recall de quedas (tolerância de 2 pontos);
3. não ter mais alarmes falsos por hora que o dobro da referência.

Se falhar, **o modelo não é exportado**. É isso que impede uma versão pior de chegar ao grupo.

In [ ]:
criterios = {
    "recall_quedas >= 0.90": metricas["recall_quedas"] >= 0.90,
    "recall >= referencia - 0.02": metricas["recall_quedas"] >= metricas_ref["recall_quedas"] - 0.02,
    "alarmes falsos <= 2x referencia": metricas["alarmes_falsos_por_hora"] <= 2 * metricas_ref["alarmes_falsos_por_hora"] + 1e-9,
}
for nome, ok in criterios.items():
    print(("PASSOU " if ok else "FALHOU ") + nome)
APROVADO = all(criterios.values())
print("\nAPROVADO para exportar" if APROVADO else "\nREPROVADO: ajuste e treine de novo")

## 8. Registrar no MLflow e exportar

O MLflow guarda parâmetros, métricas e o modelo de cada execução em `mlruns/`. Baixem a pasta junto com o modelo para comparar experimentos depois (`mlflow ui` na máquina de vocês).

A pasta `modelo_queda/` exportada tem `config_queda.json` + `modelo_queda.keras`. Ela vai para **`modelos/queda/`** na raiz do projeto.

In [ ]:
import shutil, sklearn
from google.colab import files

import keras
ambiente = {"tensorflow": tf.__version__, "keras": keras.__version__, "numpy": np.__version__,
            "sklearn": sklearn.__version__}

try:
    import mlflow
    mlflow.set_experiment("detector-queda")
    with mlflow.start_run(run_name="lstm"):
        mlflow.log_params({**HIPER, "limiar": limiar, "recall_minimo": RECALL_MINIMO,
                           "janelas_treino": int(treino.sum()), "com_apple_watch": bool(PASTA_APPLE_WATCH)})
        mlflow.log_metrics({k: float(v) for k, v in metricas.items() if np.isfinite(v)})
        mlflow.log_metrics({f"ref_{k}": float(v) for k, v in metricas_ref.items() if np.isfinite(v)})
        mlflow.set_tag("quality_gate", "aprovado" if APROVADO else "reprovado")
        comparacao.to_csv("comparacao.csv"); mlflow.log_artifact("comparacao.csv")
except Exception as erro:
    print("MLflow não registrou:", erro)

if APROVADO:
    exportar.salvar("modelo_queda", tipo="lstm", modelo=modelo, limiar=limiar, metricas=metricas,
                    media=media, desvio=desvio, ambiente=ambiente)
    shutil.make_archive("modelo_queda", "zip", "modelo_queda")
    shutil.make_archive("mlruns", "zip", "mlruns")
    files.download("modelo_queda.zip")
    files.download("mlruns.zip")
else:
    print("Nada exportado.")

## 9. Próximo passo: ligar na API

1. Descompacte `modelo_queda.zip` em `modelos/queda/` na raiz do projeto.
2. Instale na API a **mesma versão** do TensorFlow mostrada abaixo (`pip install tensorflow-cpu==<versão>`).
3. Suba a API: `python -m uvicorn app.main:app --port 8000`. O terminal deve mostrar `detector de quedas carregado: lstm ...`.
4. Teste sem relógio: `python -m scripts.simular_relogio app/rede-neural/dataset/F01/U03_R01_accel.csv`.

Passo a passo completo, com o Sensor Logger e o WhatsApp, em `docs/QUEDA.md`.

In [ ]:
print("TensorFlow usado no treino:", tf.__version__)